# SonoTrust — First medical imaging experiment
Learn to outline lesions in breast ultrasound images with a compact U-Net.

Run cells in order. Artificial-data results check software only. Real-image
results require a verified data audit and a frozen evaluation protocol.

Select Runtime → Change runtime type → T4 GPU if available. This notebook is
for educational research, not diagnosis. Download results before Colab closes.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, sys, subprocess
uploaded = files.upload()  # Select the SonoTrust project ZIP
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Upload exactly one project ZIP')
root = Path('/content/sonotrust')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as z:
    for info in z.infolist():
        if not (root / info.filename).resolve().is_relative_to(root.resolve()):
            raise ValueError('Unsafe ZIP path')
    z.extractall(root)
projects = list(root.rglob('train.py'))
if len(projects) != 1:
    raise ValueError('Expected one project folder containing train.py')
os.chdir(projects[0].parent)
print('Working directory:', Path.cwd())
def run(*args):
    subprocess.run([sys.executable, *args], check=True)
run('-m', 'pip', 'install', '-q', '-r', 'requirements.txt')
import torch
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.is_available())

## 1. Verify the software on artificial shapes
This does not produce medical research evidence.

In [ ]:
run('make_synthetic_data.py')
run('-m', 'pytest', '-q')
run('audit_data.py', '--data', 'data/synthetic/BUSBRA', '--out', 'results/synthetic_audit')
run('train.py', '--data', 'data/synthetic/BUSBRA', '--out', 'results/smoke',
    '--side', '64', '--base', '4', '--epochs', '2')
run('evaluate.py', '--data', 'data/synthetic/BUSBRA', '--out', 'results/smoke')

## 2. Download and audit the official images
Source: https://zenodo.org/records/8231412.
The download verifies the published archive checksum. The audit inspects labels,
mask geometry, case groups, split membership and exact duplicate pixels.

The case prefix is an inferred grouping unit; verify that it corresponds to
patient identity before reporting patient-level research results. Exact hashes
do not detect all near duplicates.

In [ ]:
run('download_data.py')
run('audit_data.py', '--data', 'data/BUSBRA', '--out', 'results/audit')
from IPython.display import display, Image as DisplayImage
import json
summary = json.loads(Path('results/audit/audit_summary.json').read_text())
print(json.dumps(summary, indent=2))
display(DisplayImage(filename='results/audit/training_examples.png'))

## 3. Train a first raw-image baseline
Inspect training loss and validation Dice. The best checkpoint is chosen by
validation performance. We have not evaluated the test set yet. These modest
settings establish a first baseline, not a final optimized model.

In [ ]:
run('train.py', '--data', 'data/BUSBRA', '--out', 'results/raw_first',
    '--method', 'raw', '--side', '128', '--base', '8', '--batch-size', '8',
    '--epochs', '10', '--seed', '42')
run('plot_training.py', '--out', 'results/raw_first')
import pandas as pd
import matplotlib.pyplot as plt
history = pd.read_csv('results/raw_first/training_history.csv')
fig, axes = plt.subplots(1,2,figsize=(10,3))
axes[0].plot(history.epoch, history.train_loss)
axes[0].set(xlabel='Epoch',ylabel='Training loss')
axes[1].plot(history.epoch,history.val_dice)
axes[1].set(xlabel='Epoch',ylabel='Validation Dice',ylim=(0,1))
plt.show()

## 4. Save your progress
The results archive includes your checkpoint and logs, and excludes source
images and preview panels. Keep it private while you verify case grouping.

Next experiment: settle the raw/CLAHE comparison settings using validation,
train both methods with the same seed and settings, then run final held-out
evaluation. Do not choose settings after inspecting test scores.

In [ ]:
destination = Path('/content/SonoTrust_First_Run_Results.zip')
with zipfile.ZipFile(destination,'w',zipfile.ZIP_DEFLATED) as z:
    for p in Path('results').rglob('*'):
        if p.is_file() and (p.suffix in {'.json','.csv','.pt'} or p.name == 'learning_curves.png'):
            z.write(p,p.as_posix())
files.download(str(destination))